
# 02 — Text Preprocessing and Feature Engineering

## 1. Dataset loading and preparation

This notebook transforms the titles and abstracts of 75,110 arXiv (cs.CL category) NLP publications (2021–2025) into numerical features for unsupervised clustering.

The analysis uses the filtered corpus produced in `01_data_acquisition.ipynb`. The initial preparation consists of loading the required metadata, normalising whitespace, and validating data integrity before text vectorisation.

In [2]:
import json
from pathlib import Path
import pandas as pd

# Path to the filtered corpus from Stage 1
data_path = Path("data/arxiv_cscl_2021_2025.jsonl")

if not data_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

# Select only the fields needed for the analysis
columns = [
    "id",
    "title",
    "abstract",
    "categories",
    "first_submission_date"
]

# Read the filtered JSONL file sequentially
records = []

with data_path.open("r", encoding="utf-8") as file:
    for line in file:
        paper = json.loads(line)
        records.append({
            column: paper.get(column) for column in columns
        })

df = pd.DataFrame.from_records(records, columns=columns)

print(f"Loaded documents: {len(df):,}")
print(f"DataFrame memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

display(df[["id", "title", "first_submission_date"]].head())

Loaded documents: 75,110
DataFrame memory: 112.4 MB


,id,title,first_submission_date
0,2101.00117,Multi-task Retrieval for Knowledge-Intensive T...,2021-01-01
1,2101.00121,WARP: Word-level Adversarial ReProgramming,2021-01-01
2,2101.00123,Intent Classification and Slot Filling for Pri...,2021-01-01
3,2101.00124,Discourse-level Relation Extraction via Graph ...,2021-01-01
4,2101.00130,Sensei: Self-Supervised Sensor Name Segmentation,2021-01-01


In [3]:

# Normalise whitespace without removing meaningful content
df["clean_title"] = (
    df["title"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["clean_abstract"] = (
    df["abstract"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Prepare submission dates for temporal analysis
df["first_submission_date"] = pd.to_datetime(
    df["first_submission_date"],
    errors="coerce"
)

df["year"] = df["first_submission_date"].dt.year

display(
    df[["id", "clean_title", "clean_abstract", "year"]].head()
)

,id,clean_title,clean_abstract,year
0,2101.00117,Multi-task Retrieval for Knowledge-Intensive T...,Retrieving relevant contexts from a large corp...,2021
1,2101.00121,WARP: Word-level Adversarial ReProgramming,Transfer learning from pretrained language mod...,2021
2,2101.00123,Intent Classification and Slot Filling for Pri...,Understanding privacy policies is crucial for ...,2021
3,2101.00124,Discourse-level Relation Extraction via Graph ...,The ability to capture complex linguistic stru...,2021
4,2101.00130,Sensei: Self-Supervised Sensor Name Segmentation,"A sensor name, typically an alphanumeric strin...",2021


In [4]:
# Verifing the dataset 

print("Total documents:", len(df))
print("Unique paper IDs:", df["id"].nunique())
print("Duplicate IDs:", df["id"].duplicated().sum())

print("Missing IDs:", df["id"].isna().sum())
print("Empty cleaned titles:", df["clean_title"].fillna("").eq("").sum())
print("Empty cleaned abstracts:", df["clean_abstract"].fillna("").eq("").sum())

print("Invalid dates:", df["first_submission_date"].isna().sum())
print("Outside 2021–2025:", (~df["year"].between(2021, 2025)).sum())

print(
    "Papers without cs.CL:",
    (~df["categories"].fillna("").str.split().map(
        lambda categories: "cs.CL" in categories
    )).sum()
)

print("\nDocuments by submission year:")
print(df["year"].value_counts().sort_index())

Total documents: 75110
Unique paper IDs: 75110
Duplicate IDs: 0
Missing IDs: 0
Empty cleaned titles: 0
Empty cleaned abstracts: 0
Invalid dates: 0
Outside 2021–2025: 0
Papers without cs.CL: 0

Documents by submission year:
year
2021     8083
2022     8971
2023    13603
2024    20689
2025    23764
Name: count, dtype: int64



## 2. Document representation

Titles and abstracts are concatenated to construct one textual document per paper.

This approach combines the concise topical information of titles with the richer descriptions provided by abstracts. Both fields receive equal treatment during concatenation, without introducing explicit title weighting.

The complete corpus will be used to construct a shared TF–IDF vocabulary, providing a consistent feature space for subsequent clustering and temporal comparisons.

In [7]:
# Combine cleaned titles and abstracts
df["document_text"] = (
    df["clean_title"] + " " + df["clean_abstract"]
)

# Count whitespace-separated words in each field
df["title_word_count"] = df["clean_title"].str.split().str.len()
df["abstract_word_count"] = df["clean_abstract"].str.split().str.len()

df["document_word_count"] = (
    df["title_word_count"] + df["abstract_word_count"]
)

# Estimate the proportion contributed by titles
df["title_word_share"] = (
    df["title_word_count"] / df["document_word_count"]
)

print("Documents:", len(df))
print("Empty combined documents:", df["document_text"].str.strip().eq("").sum())

print("\nAverage word counts:")
print(
    df[
        ["title_word_count", "abstract_word_count",
         "document_word_count", "title_word_share"]
    ].mean().round(3)
)

print("\nCombined document length distribution:")
print(
    df["document_word_count"].describe(
        percentiles=[0.01, 0.50, 0.99]
    ).round(1)
)

Documents: 75110
Empty combined documents: 0

Average word counts:
title_word_count        10.049
abstract_word_count    167.588
document_word_count    177.637
title_word_share         0.060
dtype: float64

Combined document length distribution:
count    75110.0
mean       177.6
std         43.3
min         20.0
1%          82.0
50%        175.0
99%        282.0
max        492.0
Name: document_word_count, dtype: float64


In [8]:

# Reproducible random sample of three papers
sample = df.sample(n=3, random_state=42)

for _, paper in sample.iterrows():
    print("=" * 80)
    print("Paper ID:", paper["id"])
    print("Year:", paper["year"])
    print("Title:", paper["clean_title"])
    print("Combined word count:", paper["document_word_count"])
    print("Title word share:", round(paper["title_word_share"], 3))
    print("\nCombined text (first 500 characters):")
    print(paper["document_text"][:500])
    print()

Paper ID: 2401.10580
Year: 2024
Title: PHOENIX: Open-Source Language Adaption for Direct Preference Optimization
Combined word count: 134
Title word share: 0.06

Combined text (first 500 characters):
PHOENIX: Open-Source Language Adaption for Direct Preference Optimization Large language models have gained immense importance in recent years and have demonstrated outstanding results in solving various tasks. However, despite these achievements, many questions remain unanswered in the context of large language models. Besides the optimal use of the models for inference and the alignment of the results to the desired specifications, the transfer of models to other languages is still an underdev

Paper ID: 2406.10984
Year: 2024
Title: Revisiting Cosine Similarity via Normalized ICA-transformed Embeddings
Combined word count: 134
Title word share: 0.052

Combined text (first 500 characters):
Revisiting Cosine Similarity via Normalized ICA-transformed Embeddings Cosine similarity is widely u

## 3. Tokenization and n-gram inspection.

In [9]:

from sklearn.feature_extraction.text import TfidfVectorizer

examples = {
    "Scientific terminology":
        "Large language models use retrieval-augmented generation "
        "and chain-of-thought reasoning.",

    "Scientific system terminology":
        "A dialogue system supports multilingual dialogue systems.",

    "Actual paper title":
        df.loc[
            df["id"] == "2401.10580", "clean_title"
        ].iloc[0]
}

for name, text in examples.items():
    print(f"\n{'=' * 70}")
    print(name)
    print("Original:", text)

    for ngram_range in [(1, 1), (1, 2), (1, 3)]:
        vectorizer = TfidfVectorizer(
            ngram_range=ngram_range,
            stop_words="english"
        )

        tokens = vectorizer.build_analyzer()(text)

        print(f"\nN-gram range {ngram_range}:")
        print(tokens)



Scientific terminology
Original: Large language models use retrieval-augmented generation and chain-of-thought reasoning.

N-gram range (1, 1):
['large', 'language', 'models', 'use', 'retrieval', 'augmented', 'generation', 'chain', 'thought', 'reasoning']

N-gram range (1, 2):
['large', 'language', 'models', 'use', 'retrieval', 'augmented', 'generation', 'chain', 'thought', 'reasoning', 'large language', 'language models', 'models use', 'use retrieval', 'retrieval augmented', 'augmented generation', 'generation chain', 'chain thought', 'thought reasoning']

N-gram range (1, 3):
['large', 'language', 'models', 'use', 'retrieval', 'augmented', 'generation', 'chain', 'thought', 'reasoning', 'large language', 'language models', 'models use', 'use retrieval', 'retrieval augmented', 'augmented generation', 'generation chain', 'chain thought', 'thought reasoning', 'large language models', 'language models use', 'models use retrieval', 'use retrieval augmented', 'retrieval augmented generatio

## 4. TF–IDF feature extraction

In [10]:
import numpy as np
import sklearn
from time import perf_counter
from sklearn.feature_extraction.text import TfidfVectorizer

# Shared baseline configuration
tfidf_params = {
    "max_features": 10000,
    "min_df": 5,
    "max_df": 0.90,
    "stop_words": "english",
    "norm": "l2",
    "dtype": np.float32
}

# Candidate configurations
configurations = {
    "Unigrams": (1, 1),
    "Unigrams + bigrams": (1, 2)
}

vectorizers = {}
matrices = {}

print("scikit-learn version:", sklearn.__version__)

for name, ngram_range in configurations.items():

    vectorizer = TfidfVectorizer(
        **tfidf_params,
        ngram_range=ngram_range
    )

    start = perf_counter()

    X = vectorizer.fit_transform(df["document_text"])

    duration = perf_counter() - start

    vectorizers[name] = vectorizer
    matrices[name] = X

    print(f"\n{name}")
    print(f"Shape: {X.shape}")
    print(f"Nonzero elements: {X.nnz:,}")
    print(f"Fit time: {duration:.1f} seconds")


scikit-learn version: 1.9.1

Unigrams
Shape: (75110, 10000)
Nonzero elements: 6,207,305
Fit time: 4.1 seconds

Unigrams + bigrams
Shape: (75110, 10000)
Nonzero elements: 7,465,323
Fit time: 15.1 seconds


In [ ]:
results = []

for name, X in matrices.items():

    vectorizer = vectorizers[name]
    features = vectorizer.get_feature_names_out()

    unigram_count = sum(" " not in term for term in features)
    bigram_count = sum(" " in term for term in features)

    sparse_memory_mb = (
        X.data.nbytes
        + X.indices.nbytes
        + X.indptr.nbytes
    ) / (1024 ** 2)

    results.append({
        "Representation": name,
        "Documents": X.shape[0],
        "Features": X.shape[1],
        "Unigrams": unigram_count,
        "Bigrams": bigram_count,
        "Nonzero values": X.nnz,
        "Density (%)": 100 * X.nnz / (X.shape[0] * X.shape[1]),
        "Avg. features/document": X.nnz / X.shape[0],
        "Empty document vectors": (X.getnnz(axis=1) == 0).sum(),
        "Sparse matrix memory (MB)": sparse_memory_mb
    })

comparison = pd.DataFrame(results).set_index("Representation")

display(comparison.round(2))

,Documents,Features,Unigrams,Bigrams,Nonzero values,Density (%),Avg. features/document,Empty document vectors,Sparse matrix memory (MB)
Representation,,,,,,,,,
Unigrams,75110,10000,10000,0,6207305,0.83,82.64,0,47.64
Unigrams + bigrams,75110,10000,5654,4346,7465323,0.99,99.39,0,57.24


In [12]:
for name, X in matrices.items():

    vectorizer = vectorizers[name]
    features = vectorizer.get_feature_names_out()

    # Mean TF-IDF weight per feature over all documents
    mean_weights = np.asarray(X.mean(axis=0)).ravel()

    top_indices = np.argsort(mean_weights)[-20:][::-1]

    print(f"\n{'=' * 60}")
    print(f"TOP 20 FEATURES: {name}")

    for idx in top_indices:
        print(f"{features[idx]:30s} {mean_weights[idx]:.5f}")

    if name == "Unigrams + bigrams":

        bigram_indices = np.array([
            i for i, term in enumerate(features)
            if " " in term
        ])

        top_bigrams = bigram_indices[
            np.argsort(mean_weights[bigram_indices])[-20:][::-1]
        ]

        print("\nTOP 20 BIGRAMS")

        for idx in top_bigrams:
            print(f"{features[idx]:30s} {mean_weights[idx]:.5f}")
        


TOP 20 FEATURES: Unigrams
models                         0.03471
language                       0.03157
llms                           0.02794
model                          0.02723
data                           0.02423
text                           0.02041
reasoning                      0.02022
based                          0.01948
large                          0.01946
tasks                          0.01857
training                       0.01857
performance                    0.01841
knowledge                      0.01828
learning                       0.01782
task                           0.01681
llm                            0.01584
generation                     0.01512
human                          0.01465
speech                         0.01454
dataset                        0.01435

TOP 20 FEATURES: Unigrams + bigrams
models                         0.03197
language                       0.02924
llms                           0.02583
model                          0.02520
